# 07. Auditoría de Tags Editoriales y Propuesta de Normalización

Analizamos las etiquetas reales (`post_tag`) de los articulos de Anfibia, extraídas de la base de datos original.
Comparamos las etiquetas  frente a los **tópicos semánticos latentes** (BERTopic) para diagnosticar problemas de cobertura y solapamiento.

In [1]:
# 1. Parche de compatibilidad obligatorio
import coverage.types as ct
for name, fallback in {
    'Tracer': getattr(ct, 'TTracer', object),
    'TShouldTraceFn': object,
    'TShouldStartContextFn': object,
    'TFileDisposition': object,
}.items():
    if not hasattr(ct, name):
        setattr(ct, name, fallback)

# 2. Imports
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from pathlib import Path
from bertopic import BERTopic
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings("ignore")

# 3. Rutas
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed" / "corpus_anfibia_lematizado.csv"
MODEL_PATH = PROJECT_ROOT / "outputs" / "bertopic_hibrido" / "bertopic_hibrido_model"

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "analisis_etiquetas"
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

# 4. Carga de datos
df = pd.read_csv(DATA_PROCESSED)
df["doc_id"] = df.index

# 5. Carga de modelo y predicción
topic_model = BERTopic.load(MODEL_PATH)
topic_info = topic_model.get_topic_info().set_index("Topic")["Name"].to_dict()

print(f"Total de notas cruzadas: {len(df):,}")

c:\Trabajos\anfibia-unsam\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total de notas cruzadas: 3,556


## 1. Cobertura Histórica de Tags

In [2]:
df.columns

Index(['doc_id', 'ID', 'source_url', 'title_text', 'tagline', 'autor_names',
       'post_date', 'post_year', 'category', 'post_tag', 'text_type',
       'document_text_final', 'document_text_original'],
      dtype='str')

In [3]:
total = len(df)
df["tiene_tag"] = df["post_tag"].notna() & (df["post_tag"].str.strip() != "")

con_tag = df["tiene_tag"].sum()
sin_tag = total - con_tag

print(f"Notas con etiquetas: {con_tag:,} ({(con_tag/total)*100:.1f}%)")
print(f"Notas SIN etiquetas: {sin_tag:,} ({(sin_tag/total)*100:.1f}%)")

fig = go.Figure(data=[go.Pie(labels=["Con Tags", "Sin Tags"], values=[con_tag, sin_tag], hole=.4)])
fig.update_layout(title="Cobertura de Etiquetas en el Archivo", width=500, height=400)
fig.show()

Notas con etiquetas: 1,589 (44.7%)
Notas SIN etiquetas: 1,967 (55.3%)


In [4]:
# Agrupamos por año: Total de notas vs Notas con tag
evolucion = (
    df.groupby("post_year")
    .agg(
        total_notas=("doc_id", "count"),
        notas_con_tag=("tiene_tag", "sum")
    )
    .reset_index()
)
evolucion["notas_sin_tag"] = evolucion["total_notas"] - evolucion["notas_con_tag"]
evolucion["pct_con_tag"] = (evolucion["notas_con_tag"] / evolucion["total_notas"] * 100).round(1)
evolucion["pct_sin_tag"] = (evolucion["notas_sin_tag"] / evolucion["total_notas"] * 100).round(1)

# Filtramos posibles años nulos o erróneos
evolucion = evolucion[evolucion["post_year"] > 2010]

# 3. Gráfico de barras apiladas
fig_evo = go.Figure()

# Texto condicional: oculta el porcentaje si la barra es menor al 6% para evitar desbordes
textos_verdes = [f"{p}%" if p > 6 else "" for p in evolucion["pct_con_tag"]]

# Barra inferior: Notas que SÍ tienen tag
fig_evo.add_trace(go.Bar(
    x=evolucion["post_year"],
    y=evolucion["notas_con_tag"],
    name="Con Etiquetas",
    marker_color='#2ca02c',
    text=textos_verdes,
    textposition='inside',
    textangle=0,  # <--- ACÁ ESTÁ EL CAMBIO: 0 para texto horizontal
    insidetextanchor='middle',
    customdata=evolucion["pct_con_tag"],
    hovertemplate="<b>%{x}</b><br>Con etiquetas: %{y} notas (%{customdata}%)<extra></extra>"
))

# Barra superior: Notas SIN tag
fig_evo.add_trace(go.Bar(
    x=evolucion["post_year"],
    y=evolucion["notas_sin_tag"],
    name="Sin Etiquetas (Vacío)",
    marker_color='#d62728',
    customdata=evolucion["pct_sin_tag"],
    hovertemplate="<b>%{x}</b><br>Sin etiquetas: %{y} notas (%{customdata}%)<extra></extra>"
))

fig_evo.update_layout(
    barmode='stack',
    title="<b>Notas con y sin Tags por Año</b>",
    xaxis_title="Año de Publicación",
    yaxis_title="Cantidad de Artículos",
    height=500,
    width=900,
    plot_bgcolor='rgba(0,0,0,0)',
    legend=dict(orientation="h", yanchor="bottom", y=-0.2, xanchor="center", x=0.5)
)
fig_evo.update_xaxes(type='category')
fig_evo.write_html(OUTPUT_DIR / "03_evolucion_etiquetado.html")
fig_evo.show()

## 2. Fragmentación y Uso Único de Etiquetas
Separamos las etiquetas múltiples (divididas por `;`) y analizamos su frecuencia.

In [5]:
TOP = 15
# Desanidar tags separados por punto y coma
tags_list = (
    df["post_tag"]
    .dropna()
    .str.split(r";")
    .explode()
    .str.strip()
    .loc[lambda x: x != ""]
)

tag_counts = tags_list.value_counts()
tags_unicos = len(tag_counts)
tags_una_vez = (tag_counts == 1).sum()

print(f"Cantidad de tags únicos creados: {tags_unicos:,}")
print(f"Tags usados en 1 sola nota: {tags_una_vez:,} ({(tags_una_vez/tags_unicos)*100:.1f}%)")

print(f"\nTop {TOP} Tags más usados:")
display(tag_counts.head(TOP).to_frame("Frecuencia"))

Cantidad de tags únicos creados: 881
Tags usados en 1 sola nota: 444 (50.4%)

Top 15 Tags más usados:


,Frecuencia
post_tag,
Internacional,238
Javier Milei,162
Derechas,114
Zapping cultural,84
Derechos Humanos,77
UNSAM,67
Chile,65
Feminismos,62
Peronismo,61


In [6]:
# ==============================================================================
# 2.3. Muestreo y Diagnóstico de Tags Unicos
# ==============================================================================

# 1. Asegurar la serie de tags limpios
tags_series = df["post_tag"].dropna().str.split(r"[;,]").explode().str.strip()
tags_series = tags_series[tags_series != ""]

# 2. Filtrar tags que aparecen exactamente 1 sola vez
counts = tags_series.value_counts()
unicos = counts[counts == 1].index.to_series().reset_index(drop=True)

df_diag = pd.DataFrame({"tag": unicos})
df_diag["cant_palabras"] = df_diag["tag"].str.split().str.len()

# 3. Distribución por cantidad de palabras
print("Distribución de longitud (palabras por tag unico):")
for n_words, total in df_diag["cant_palabras"].value_counts().sort_index().items():
    pct = (total / len(df_diag)) * 100
    print(f" • {n_words} palabra(s): {total} tags ({pct:.1f}%)")

print("\n" + "=" * 60)
print(f"Muestra aleatoria de 30 tags Unicos (de un total de {len(df_diag)}):")
print("=" * 60)

# 4. Muestra representativa
muestra = df_diag["tag"].sample(n=min(30, len(df_diag)), random_state=42).tolist()
for i, t in enumerate(muestra, 1):
    print(f"{i:2d}. {t}")

Distribución de longitud (palabras por tag unico):
 • 1 palabra(s): 232 tags (52.3%)
 • 2 palabra(s): 159 tags (35.8%)
 • 3 palabra(s): 39 tags (8.8%)
 • 4 palabra(s): 11 tags (2.5%)
 • 5 palabra(s): 3 tags (0.7%)

Muestra aleatoria de 30 tags Unicos (de un total de 444):
 1. Nora Dalmasso
 2. cancelación
 3. Honduras
 4. Canal de Panamá
 5. Xiomara Castro
 6. Autista
 7. Familiares
 8. Noche
 9. Pep
10. Imaginación política
11. Ciudades
12. Amistad
13. #EAAF
14. Inteligencia
15. Ecología
16. gendarmería
17. Derechos sociales
18. Disidencias sexuales
19. Moda
20. Beca crónica
21. Randazzo
22. Animales
23. Palito Ortega
24. Liliana Herrero
25. Instagram
26. Vínculos
27. Invasión
28. Ríos entubados
29. Fundamentalistas del Aire Acondicionado
30. Soledad Acuña


## 3. Etiquetas en las categorias

Recordamos como se distribuyen las categorias.

In [7]:
## 2.1 Distribución de Categorías Editoriales
import re

# 1. Normalizar y explotar manteniendo el dataframe listo
df_cat_exploded = (
    df[["ID", "post_year", "category", "post_tag"]]
    .copy()
    .assign(
        categoria_normalizada=lambda d: d["category"]
        .fillna("Sin categoria")
        .astype(str)
        .apply(lambda x: [p.strip() for p in re.split(r"[;,]", x) if p.strip()] or ["Sin categoria"])
    )
    .explode("categoria_normalizada")
    .reset_index(drop=True)
)

# 2. Conteo de artículos por categoría
df_cat_dist = (
    df_cat_exploded.groupby("categoria_normalizada")["ID"]
    .nunique()
    .reset_index(name="notas")
    .sort_values("notas", ascending=True)
)

# 3. Gráfico breve
fig_cat = px.bar(
    df_cat_dist, x="notas", y="categoria_normalizada", orientation="h",
    title="<b>Distribución de Notas por Categoría (Universo Lematizado)</b>",
    text="notas", color="notas", color_continuous_scale="Blues"
)
fig_cat.update_traces(textposition="outside")
fig_cat.update_layout(
    height=400, plot_bgcolor="rgba(0,0,0,0)", paper_bgcolor="#fff",
    coloraxis_showscale=False, xaxis_title="", yaxis_title="",
    margin=dict(l=150, r=40, t=50, b=30)
)
fig_cat.write_html(OUTPUT_DIR / "03a_distribucion_categorias.html")
fig_cat.show()

Mostramos como se distribuyen N-TOP etiquetas sobre las categorias.

In [8]:
TOP = 15

# 1. Desanidar los tags directamente sobre df_cat_exploded
df_tags_cat = df_cat_exploded.dropna(subset=["post_tag"]).copy()
df_tags_cat["tag"] = df_tags_cat["post_tag"].str.split(r"[;,]")
df_tags_cat = df_tags_cat.explode("tag").reset_index(drop=True)
df_tags_cat["tag"] = df_tags_cat["tag"].str.strip().str.title()
df_tags_cat = df_tags_cat[df_tags_cat["tag"] != ""]

# 2. Top N global y cálculo de totales por tag (artículos únicos)
tag_counts = df_tags_cat.groupby("tag")["ID"].nunique().sort_values(ascending=False).head(TOP)
top_nombres = tag_counts.index.tolist()

mapeo_etiqueta_total = {tag: f"{tag} ({conteo})" for tag, conteo in tag_counts.items()}
orden_filas = [mapeo_etiqueta_total[tag] for tag in top_nombres][::-1]

# 3. Filtrar y formatear
df_top = df_tags_cat[df_tags_cat["tag"].isin(top_nombres)].reset_index(drop=True)
df_top["tag_label"] = df_top["tag"].map(mapeo_etiqueta_total)

# 4. Matriz de Frecuencia (Tag vs Categoría Normalizada)
matriz = pd.crosstab(df_top["tag_label"], df_top["categoria_normalizada"])
matriz = matriz.reindex(orden_filas)
matriz = matriz.loc[:, matriz.sum(axis=0) > 0]

# 5. Render del Heatmap
fig_heatmap = px.imshow(
    matriz,
    labels=dict(x="Categoría Editorial", y="", color="Notas"),
    x=matriz.columns,
    y=matriz.index,
    color_continuous_scale="Blues",
    aspect="auto",
    text_auto=True
)

fig_heatmap.update_traces(
    hovertemplate="<b>Tag:</b> %{y}<br><b>Categoría:</b> %{x}<br><b>Notas en sección:</b> %{z}<extra></extra>"
)

fig_heatmap.update_layout(
    title=f"<b>Top {TOP} Etiquetas (Total de Notas) vs Categorías Editoriales</b>",
    xaxis_title="",
    yaxis_title="",
    height=max(500, TOP * 32),
    plot_bgcolor="rgba(0,0,0,0)",
    paper_bgcolor="#fff",
    margin=dict(l=180, b=100)
)
fig_heatmap.update_xaxes(tickangle=-45)

fig_heatmap.write_html(OUTPUT_DIR / f"03b_top{TOP}_tags_heatmap.html")
fig_heatmap.show()

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import numpy as np

# ==============================================================================
# 2.3. Donas por Categoría con Dropdown Temporal
# ==============================================================================

TOP_TAGS_POR_CAT = 10 
MIN_NOTAS_CAT = 20

# 1. Categorías válidas basadas en artículos únicos totales
notas_totales_por_cat = df_cat_exploded[df_cat_exploded["categoria_normalizada"] != "Sin categoria"].groupby("categoria_normalizada")["ID"].nunique()
categorias_validas = notas_totales_por_cat[notas_totales_por_cat >= MIN_NOTAS_CAT].sort_values(ascending=False).index.tolist()

años_disponibles = sorted([int(a) for a in df_cat_exploded["post_year"].dropna().unique() if a > 2010])
pasos_temporales = ["Global"] + [str(a) for a in años_disponibles]

n_cats = len(categorias_validas)
n_cols = 3
n_rows = int(np.ceil(n_cats / n_cols))

# Títulos limpios, solo el nombre de la categoría
titulos_iniciales = [f"<b>{cat}</b>" for cat in categorias_validas]

fig_donuts = make_subplots(
    rows=n_rows,
    cols=n_cols,
    specs=[[{"type": "domain"}] * n_cols for _ in range(n_rows)],
    subplot_titles=titulos_iniciales,
    vertical_spacing=0.18,
    horizontal_spacing=0.12
)

colores_anillo = px.colors.qualitative.Set2

def formatear_label(txt, max_len=14):
    if len(txt) > max_len:
        partes = txt.split(" ")
        if len(partes) > 1:
            medio = len(partes) // 2
            return " ".join(partes[:medio]) + "<br>" + " ".join(partes[medio:])
    return txt

# 2. Generar trazas para cada período
for paso_idx, paso in enumerate(pasos_temporales):
    es_global = (paso == "Global")
    
    if es_global:
        df_tags_periodo = df_tags_cat
    else:
        df_tags_periodo = df_tags_cat[df_tags_cat["post_year"] == int(paso)]

    es_visible_inicial = (paso_idx == 0)

    for idx, cat in enumerate(categorias_validas):
        r = (idx // n_cols) + 1
        c = (idx % n_cols) + 1

        df_c = df_tags_periodo[df_tags_periodo["categoria_normalizada"] == cat]
        top_tags = df_c["tag"].value_counts().head(TOP_TAGS_POR_CAT)

        if len(top_tags) > 0:
            labels_originales = top_tags.index.tolist()
            values = top_tags.values.tolist()
            labels_formateados = [formatear_label(l) for l in labels_originales]
            colores = colores_anillo[:len(labels_formateados)]
        else:
            labels_originales = ["Sin etiquetas en el período"]
            values = [1]
            labels_formateados = ["Sin tags"]
            colores = ["#f0f0f0"]

        fig_donuts.add_trace(
            go.Pie(
                labels=labels_formateados,
                customdata=labels_originales,
                values=values,
                hole=0.60,
                name=f"{cat} ({paso})",
                marker=dict(colors=colores, line=dict(color='#ffffff', width=2)),
                textinfo="label+percent" if len(top_tags) > 0 else "none",
                textposition="outside",
                hovertemplate="<b>%{customdata}</b><br>Artículos con este tag: %{value} (%{percent})<extra></extra>" if len(top_tags) > 0 else "<b>Sin tags registrados</b><extra></extra>",
                showlegend=False,
                visible=es_visible_inicial
            ),
            row=r,
            col=c
        )

# 3. Botones del Dropdown
total_trazas = len(fig_donuts.data)
trazas_por_paso = n_cats

dropdown_buttons = []
for i, paso in enumerate(pasos_temporales):
    visibilidad = [False] * total_trazas
    inicio = i * trazas_por_paso
    fin = inicio + trazas_por_paso
    visibilidad[inicio:fin] = [True] * trazas_por_paso

    dropdown_buttons.append(dict(
        method="update",
        args=[
            {"visible": visibilidad},
            {"title": f"<b>Distribución de Tags por Categoría (Top {TOP_TAGS_POR_CAT}) — Período: {paso}</b>"}
        ],
        label=f"  {paso}  "
    ))

updatemenus = [dict(
    buttons=dropdown_buttons,
    direction="down",
    pad={"r": 10, "t": 10},
    showactive=True,
    x=0.0,
    xanchor="left",
    y=1.12,
    yanchor="top",
    bgcolor="#ffffff",
    bordercolor="#cccccc",
    font=dict(size=12, color="#333333")
)]

fig_donuts.update_layout(
    updatemenus=updatemenus,
    title_text=f"<b>Distribución de Tags por Categoría (Top {TOP_TAGS_POR_CAT}) — Período: Global</b>",
    height=480 * n_rows + 40,
    width=1650,
    plot_bgcolor="rgba(0,0,0,0)",
    paper_bgcolor="#fff",
    margin=dict(t=120, b=40, l=60, r=60),
    font=dict(size=11)
)

fig_donuts.write_html(OUTPUT_DIR / "03c_tags_donas_dropdown_temporal.html")
fig_donuts.show()

## 3. Solapamiento: Tags vs Tópicos BERTopic
Vemos cómo un mismo tópico semántico fue etiquetado con múltiples tags manuales distintos.

In [11]:
# 5. Carga de modelo y predicción
topic_model = BERTopic.load(MODEL_PATH)
topic_info = topic_model.get_topic_info().set_index("Topic")["Name"].to_dict()

print("Asignando tópicos a los documentos (esto puede demorar un minuto)...")
topics, _ = topic_model.transform(df["document_text_final"].fillna("").astype(str).tolist())
df["topic"] = topics
df["topic_name"] = df["topic"].map(topic_info)


# Armamos df desanidado completo para agrupar
df_desanidado = df.drop("post_tag", axis=1).join(
    df["post_tag"].dropna().str.split(r";").explode().str.strip().rename("tag")
).dropna(subset=["tag"])

df_desanidado = df_desanidado[df_desanidado["tag"] != ""]

# Agrupamos por tópico
solapamiento = (
    df_desanidado[df_desanidado["topic"] != -1]
    .groupby(["topic", "topic_name"])
    .agg(
        notas_con_tag=("doc_id", "nunique"),
        tags_unicos_usados=("tag", "nunique"),
        ejemplos_tags=("tag", lambda x: " | ".join(pd.Series.unique(x)[:5]))
    )
    .reset_index()
    .sort_values("notas_con_tag", ascending=False)
)

display(solapamiento.head(10))
solapamiento.to_csv(OUTPUT_DIR / "auditoria_tags_por_topico.csv", index=False)

Asignando tópicos a los documentos (esto puede demorar un minuto)...


,topic,topic_name,notas_con_tag,tags_unicos_usados,ejemplos_tags
0,0,0_político_política_electoral_candidato,113,144,Escuela de Política y Gobierno | UNSAM | Brexi...
1,1,1_feminismo_feminista_patriarcado_masculinidad,63,66,#NiUnaMenos | LGBTIQ+ | Orgullo | Memoria tran...
2,2,2_fifa_futbolista_fútbol_deporte,50,54,Perfil político | Barras bravas | Boca Juniors...
4,4,4_déficit_deuda_endeudamiento_inflación,43,46,Fondos buitre | Movimientos sociales | Violenc...
5,5,5_pedagógico_enseñanza_escolar_educativo,38,51,Encuentro Anfibio | Feria del Libro | Día del ...
18,18,18_chileno_chile_político_dictadura,36,27,Diego Maradona | Chile | Estallido en Chile | ...
13,13,13_papal_catolicismo_arzobispo_católica,32,29,Papa Francisco | Rossana Reguillo | Catolicism...
10,10,10_boliviano_bolivia_político_política,30,43,Catolicismo | Papa Francisco | Paraguay | Reli...
16,16,16_brasil_brasilero_brasilia_brasileño,30,16,Brasil | Dilma Rousseff | Internacional | Mund...
8,8,8_epidemia_vacunar_virus_vacuna,23,24,Rosariocronica | Rosario | Covid-19 | Escuela ...


In [19]:
import plotly.express as px

TOP_TAGS = 30
TOP_TOPICOS = 10

# 1. Copiamos y estandarizamos tags
df_tags_topics = df_desanidado[df_desanidado["topic"] != -1].copy()
df_tags_topics["tag"] = df_tags_topics["tag"].str.strip().str.title()

# 2. Identificar el Top N de etiquetas manuales más usadas
tag_counts = df_tags_topics.groupby("tag")["doc_id"].nunique().sort_values(ascending=False).head(TOP_TAGS)
top_nombres_tags = tag_counts.index.tolist()

mapeo_etiqueta_total = {tag: f"{tag} ({conteo})" for tag, conteo in tag_counts.items()}
orden_filas = [mapeo_etiqueta_total[tag] for tag in top_nombres_tags][::-1]

# 3. Identificar el Top N de tópicos semánticos con más notas
topicos_counts = df_tags_topics.groupby("topic_name")["doc_id"].nunique().sort_values(ascending=False).head(TOP_TOPICOS)
top_nombres_topics = topicos_counts.index.tolist()

# 4. Filtrar el DataFrame cruzado solo con esos topes
df_top = df_tags_topics[
    df_tags_topics["tag"].isin(top_nombres_tags) & 
    df_tags_topics["topic_name"].isin(top_nombres_topics)
].reset_index(drop=True)

df_top["tag_label"] = df_top["tag"].map(mapeo_etiqueta_total)

# 5. Matriz de Frecuencia cruzada y acotada
matriz = pd.crosstab(df_top["tag_label"], df_top["topic_name"])
matriz = matriz.reindex(index=orden_filas, columns=top_nombres_topics)
matriz = matriz.loc[:, matriz.sum(axis=0) > 0] # Limpiar columnas vacías si quedaron

# 6. Render del Heatmap acotado
fig_heatmap = px.imshow(
    matriz,
    labels=dict(x="Tópico Semántico (BERTopic)", y="", color="Notas"),
    x=matriz.columns,
    y=matriz.index,
    color_continuous_scale="Blues",
    aspect="auto",
    text_auto=True
)

fig_heatmap.update_traces(
    hovertemplate="<b>Tag:</b> %{y}<br><b>Tópico:</b> %{x}<br><b>Notas solapadas:</b> %{z}<extra></extra>"
)

fig_heatmap.update_layout(
    title=f"<b>Top {TOP_TAGS} Etiquetas vs Top {TOP_TOPICOS} Tópicos Semánticos</b>",
    xaxis_title="",
    yaxis_title="",
    height=max(600, TOP_TAGS * 32),
    plot_bgcolor="rgba(0,0,0,0)",
    paper_bgcolor="#fff",
    margin=dict(l=180, b=180)
)
fig_heatmap.update_xaxes(tickangle=-45)

fig_heatmap.write_html(OUTPUT_DIR / f"04_top{TOP_TAGS}_tags_vs_top{TOP_TOPICOS}_topicos_heatmap.html")
fig_heatmap.show()

Cruzemos las Notas vs la Cantidad de Tags Únicos. Y ademas calculemos un "Índice de Fragmentación" (cuántos tags se crean por cada nota) que sirva como una tercera dimensión (color): 

si la burbuja es roja, significa que los editores inventaron casi un tag nuevo por cada artículo de ese tema.

In [20]:
import plotly.express as px

MOSTRAR_POLITICA = False

# 1. Calculamos el ratio de fragmentación sobre solapamiento
solapamiento["ratio_fragmentacion"] = (solapamiento["tags_unicos_usados"] / solapamiento["notas_con_tag"]).round(2)

# 2. Filtramos tópicos con volumen suficiente y aplicamos la flag para el Tópico 0
df_ranking = solapamiento[solapamiento["notas_con_tag"] >= 15].copy()

if not MOSTRAR_POLITICA:
    df_ranking = df_ranking[df_ranking["topic"] != 0]

# 3. Ordenamos por ratio de fragmentación (ascendente para que el peor quede arriba en barras horizontales)
df_ranking = df_ranking.sort_values("ratio_fragmentacion", ascending=True).tail(15)

# 4. Bar Chart Horizontal
fig_ranking = px.bar(
    df_ranking,
    x="ratio_fragmentacion",
    y="topic_name",
    orientation="h",
    color="ratio_fragmentacion",
    color_continuous_scale="Reds",
    text="tags_unicos_usados",
    hover_data={
        "topic": True,
        "notas_con_tag": True,
        "tags_unicos_usados": True,
        "ratio_fragmentacion": False,
        "ejemplos_tags": True,
    },
    title="<b>Proporción de etiquetas por artículo</b><br><sup>El número en la barra indica la cantidad total de tags únicos inventados para ese tema.</sup>",
    labels={
        "ratio_fragmentacion": "Índice de Caos (Tags únicos / Notas)",
        "topic_name": "Palabras del Tópico",
    },
    height=650,
    template="plotly_white",
)

# Ajustes de visualización y margen
fig_ranking.update_traces(textposition="outside")
fig_ranking.update_layout(
    xaxis_range=[0, df_ranking["ratio_fragmentacion"].max() * 1.15],
    coloraxis_showscale=False,
)

# 5. Exportar y mostrar
fig_ranking.write_html(OUTPUT_DIR / "05_ranking_caos_etiquetas.html")
fig_ranking.show()

## 4. Solapamiento de etiquetas



### Solapamiento por Coocurrencia en Documentos (Índice de Jaccard)

No se emplean modelos de Deep Learning ni redes neuronales en este punto. El análisis es puramente cuantitativo, determinista y estructurado en dos fases:
* **Matriz Binaria Documento $\times$ Tag:** Cada artículo y cada etiqueta se cruzan en una tabla booleana donde $1$ indica presencia y $0$ ausencia.
* **Producto Punto Matricial (`.dot()`):** Se multiplica la matriz por su transpuesta para calcular de forma masiva cuántos artículos comparten simultáneamente cada par de etiquetas.

### Índice de Jaccard
Para evitar que una etiqueta gigantesca absorba a todas las demás por frecuencia, utilizamos el **Índice de Jaccard**, que mide la similitud y diversidad de conjuntos finitos. Su fórmula formal es:

$$J(A, B) = \frac{|A \cap B|}{|A \cup B|} = \frac{\text{Notas Compartidas}}{\text{Total de Notas Únicas entre ambos Tags}}$$

* **Intersección ($A \cap B$):** La cantidad exacta de artículos que llevan puestos *ambos* tags al mismo tiempo.
* **Unión ($A \cup B$):** El volumen total de notas únicas abarcadas por el Tag A, el Tag B o por ambos simultáneamente (calculado como: $\text{Total}_A + \text{Total}_B - \text{Intersección}$).
* **Resultado:** Un coeficiente normalizado entre $0$ (cero solapamiento) y $1$ (idéntico comportamiento editorial).

### Parámetros de Corte y Filtro
* **Umbral de Jaccard ($> 0.15$):** Exige que al menos el 15% del universo combinado de notas esté efectivamente compartido entre ambos tags.
* **Mínimo de Coocurrencia ($\ge 3$ notas):** Descarta falsos positivos causados por coincidencias marginales o errores tipográficos aislados.

In [49]:

UMBRAL_JACCARD = 0.15          # Índice mínimo de solapamiento (0 a 1)
MIN_NOTAS_COMPARTIDAS = 3      # Cantidad mínima de artículos compartidos

# 1. Limpiar y asegurar unicidad estricta por documento y tag
df_binario = df_desanidado[["doc_id", "tag"]].copy()
df_binario["tag"] = df_binario["tag"].str.strip().str.title()

# Eliminar cualquier duplicado exacto
df_binario = df_binario.drop_duplicates(subset=["doc_id", "tag"])

# Asignar un valor constante de 1 para la matriz binaria
df_binario["presente"] = 1

# 2. Usar pivot_table en lugar de crosstab para manejar de forma segura los índices
matriz_docs_tags = df_binario.pivot_table(
    index="doc_id", 
    columns="tag", 
    values="presente", 
    fill_value=0
)

# 3. Calcular coocurrencia (notas que comparten ambos tags)
coocurrencia = matriz_docs_tags.T.dot(matriz_docs_tags)

# 4. Calcular Índice de Jaccard (intersección sobre unión)
suma_tags = matriz_docs_tags.sum(axis=0)
union = suma_tags.values[:, None] + suma_tags.values[None, :] - coocurrencia.values
jaccard = coocurrencia.values / np.where(union == 0, 1, union)

df_jaccard = pd.DataFrame(jaccard, index=coocurrencia.index, columns=coocurrencia.columns)

# 5. Extraer pares con solapamiento real en las notas
pares_documentos = []
tags_cols = df_jaccard.columns
n_tags = len(tags_cols)

for i in range(n_tags):
    for j in range(i + 1, n_tags):
        score = df_jaccard.iloc[i, j]
        compartidas = coocurrencia.iloc[i, j]
        
        # Aplicamos los parámetros definidos arriba
        if score > UMBRAL_JACCARD and compartidas >= MIN_NOTAS_COMPARTIDAS:
            pares_documentos.append({
                "Tag A": tags_cols[i],
                "Tag B": tags_cols[j],
                "Notas_Compartidas": int(compartidas),
                "Indice_Solapamiento_Jaccard": round(float(score), 3)
            })

df_documentos = pd.DataFrame(pares_documentos).sort_values(by="Indice_Solapamiento_Jaccard", ascending=False)

print(f"Pares con solapamiento en las notas publicadas: {len(df_documentos):,}")
df = df_documentos.sort_values(by='Notas_Compartidas')
display(df_documentos.head(60))


df_documentos.to_csv(OUTPUT_DIR / "tags_solapamiento_documentos.csv", index=False)

Pares con solapamiento en las notas publicadas: 107


,Tag A,Tag B,Notas_Compartidas,Indice_Solapamiento_Jaccard
15,Adolescencia,Salud Sexual Y Reproductiva,3,1.000
14,Adolescencia,Maternidad Temprana,3,1.000
12,Adolescencia,Anticoncepción,3,1.000
20,Anticoncepción,Maternidad Temprana,3,1.000
21,Anticoncepción,Salud Sexual Y Reproductiva,3,1.000
89,Maternidad Temprana,Salud Sexual Y Reproductiva,3,1.000
67,Francia Márquez,Gustavo Petro,3,0.750
80,Israel,Palestina,9,0.600
3,2001,Crisis 2001,10,0.588
45,Cop26,Crisis Climática,7,0.583


### Solapamiento semántico

Pero
Cantidad de tags únicos creados: 881
Tags usados en 1 sola nota: 444 (50.4%)


Por lo tanto, utilizamos la **matriz de similitud semántica por *embeddings*** calculada  con `SentenceTransformer`. 

Con esto evaluamos el espacio vectorial de todo el vocabulario sin importar su frecuencia, el modelo de lenguaje detecta si un error tipográfico, un sinónimo menor o una variante etiqueta núcleo.

In [53]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer

# 1. Limpiar y estandarizar uniformemente todo el dataframe de trabajo
df_clean = df_desanidado[["doc_id", "tag"]].copy()
df_clean["tag"] = df_clean["tag"].str.strip().str.title()
df_clean = df_clean[df_clean["tag"].notna() & (df_clean["tag"] != "") & (df_clean["tag"].str.lower() != "nan")]

# 2. Obtener frecuencias y catálogos sobre el set 100% limpio
tag_freqs = df_clean["tag"].value_counts()
tags_unicos = list(tag_freqs.index)

tags_unicos_sospechosos = tag_freqs[tag_freqs == 1].index.tolist()
tags_nucleo = tag_freqs[tag_freqs > 1].index.tolist()

print(f"Total de tags únicos limpios: {len(tags_unicos):,}")
print(f"Tags de uso único (huérfanos): {len(tags_unicos_sospechosos):,}")
print(f"Tags de núcleo (frecuencia > 1): {len(tags_nucleo):,}")

# 3. Asegurar que el modelo de embeddings esté cargado
if 'embedding_model' not in globals():
    embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# Generar embeddings para todos los tags únicos limpios
tag_embeddings = embedding_model.encode(tags_unicos, show_progress_bar=True)

# 4. Crear diccionario de mapeo seguro (Tag -> Índice en la matriz de embeddings)
tag_to_idx = {tag: idx for idx, tag in enumerate(tags_unicos)}

# 5. Calcular similitud coseno entre los huérfanos y el núcleo oficial
indices_huerfanos = [tag_to_idx[t] for t in tags_unicos_sospechosos]
indices_nucleo = [tag_to_idx[t] for t in tags_nucleo]

matriz_huerfanos_vs_nucleo = cosine_similarity(
    tag_embeddings[indices_huerfanos],
    tag_embeddings[indices_nucleo]
)

# 6. Encontrar el mejor match para cada tag huérfano
sugerencias_fusion = []
for i, tag_huerfano in enumerate(tags_unicos_sospechosos):
    mejor_idx_local = matriz_huerfanos_vs_nucleo[i].argmax()
    mejor_score = matriz_huerfanos_vs_nucleo[i][mejor_idx_local]
    mejor_match = tags_nucleo[mejor_idx_local]
    
    sugerencias_fusion.append({
        "Tag_Unico": tag_huerfano,
        "Sugerencia_Fusion": mejor_match,
        "Similitud_Semantica": round(float(mejor_score), 3)
    })

df_sugerencias = pd.DataFrame(sugerencias_fusion).sort_values(by="Similitud_Semantica", ascending=False)

print(f"\nSe procesaron las sugerencias para los {len(df_sugerencias):,} tags huérfanos.")
display(df_sugerencias.head(60))

df_sugerencias.to_csv(OUTPUT_DIR / "tags_huerfanos_sugerencias_fusion.csv", index=False)

Total de tags únicos limpios: 881
Tags de uso único (huérfanos): 444
Tags de núcleo (frecuencia > 1): 437


Batches: 100%|██████████| 28/28 [00:01<00:00, 19.10it/s]


Se procesaron las sugerencias para los 444 tags huérfanos.


,Tag_Unico,Sugerencia_Fusion,Similitud_Semantica
252,Mexico,México,0.999
229,Iran,Irán,0.998
227,Crisiis Climática,Crisis Climática,0.993
221,Incendio,Incendios,0.989
380,Nicolás Maduro,Nicolas Maduro,0.988
215,Laboratorio Periodismo Performàtico,Laboratorio De Periodismo Performático,0.988
344,Policiriss,Policiales,0.985
327,Océano,Océanos,0.985
428,Culturas,Cultura,0.975
29,Crónicas,Crónica,0.964


Muchos tags aparecen en una sola nota o en muy pocas.

Implementamos un análisis basado en **distribuciones de probabilidad suave (*Soft Topic Distributions*)** mediante BERTopic.

Para cada articulo, realizar tres pasos:

1. **Distribución de Tópicos por Documento (`approximate_distribution`):** El modelo calcula un vector de probabilidad continua que indica qué tan alineada está una nota con *todos* los tópicos.
   
2. **Perfil Temático Agregado por Tag:** Cruzamos estas probabilidades con las etiquetas que posee la nota y calculamos el promedio de los vectores para cada tag.
   
3. **Similitud Coseno Vectorial:** Aplicamos producto escalar normalizado entre los perfiles temáticos de las etiquetas. 

In [64]:
# ==============================================================================
# SOLAPAMIENTO DE TAGS POR PERFIL DE TÓPICOS (Recargando corpus original)
# ==============================================================================

# 1. Cargar explícitamente el corpus original para evitar colisiones de variables
df_corpus = pd.read_csv(DATA_PROCESSED)
df_corpus["doc_id"] = df_corpus.index

textos_docs = df_corpus["document_text_final"]

# 3. Calcular la distribución suave de tópicos con BERTopic
print("Calculando distribuciones de probabilidad de tópicos con BERTopic...")
doc_topic_dist, _ = topic_model.approximate_distribution(textos_docs, use_embedding_model=False)

df_dist = pd.DataFrame(doc_topic_dist)
df_dist["doc_id"] = df_corpus["doc_id"].values

# 4. Unir la distribución de tópicos con el dataframe de tags desanidado
col_topics = [c for c in df_dist.columns if c != "doc_id"]
df_tags_dist = df_desanidado[["doc_id", "tag"]].merge(df_dist, on="doc_id", how="inner")
df_tags_dist["tag"] = df_tags_dist["tag"].str.strip().str.title()

df_tags_dist = df_tags_dist[
    df_tags_dist["tag"].notna() 
    & (df_tags_dist["tag"] != "") 
    & (df_tags_dist["tag"].str.lower() != "nan")
].copy()

# 5. Agrupar por tag y promediar sus vectores de probabilidad de tópicos
tag_topic_profiles = df_tags_dist.groupby("tag")[col_topics].mean()

tags_array = tag_topic_profiles.index.values
matriz_perfiles = tag_topic_profiles.values

print(f"\nCalculando similitud coseno por perfil probabilístico para {len(tags_array):,} tags totales...")

# 6. Calcular Similitud Coseno
sim_matrix_topic_probs = cosine_similarity(matriz_perfiles)

# 7. Filtrar pares con alta similitud temática real
UMBRAL_COSENO_PROB = 0.85
pares_prob = []

n = len(tags_array)
for i in range(n):
    for j in range(i + 1, n):
        score = sim_matrix_topic_probs[i, j]
        if UMBRAL_COSENO_PROB <= score < 0.999:
            pares_prob.append({
                "Tag A": tags_array[i],
                "Tag B": tags_array[j],
                "Similitud_Coseno_Prob_Topicos": round(float(score), 3)
            })

df_pares_prob = pd.DataFrame(pares_prob).sort_values(by="Similitud_Coseno_Prob_Topicos", ascending=False)

print(f"Pares con solapamiento temático real (con probabilidades BERTopic): {len(df_pares_prob):,}")
display(df_pares_prob.head(40))

df_pares_prob.to_csv(OUTPUT_DIR / "tags_solapamiento_prob_topicos.csv", index=False)

Calculando distribuciones de probabilidad de tópicos con BERTopic...

Calculando similitud coseno por perfil probabilístico para 881 tags totales...
Pares con solapamiento temático real (con probabilidades BERTopic): 5,981


,Tag A,Tag B,Similitud_Coseno_Prob_Topicos
5335,Messi,Scaloneta,0.999
4516,Iran,Irán,0.998
5334,Messi,Qatar 2022,0.997
5838,Qatar 2022,Scaloneta,0.997
4533,Israel,Palestina,0.996
4421,Iglesia Católica,Papa Francisco,0.996
4602,Javier Milei,Sin Control 2,0.995
2133,Cop26,Crisis Climática,0.995
4528,Irán,Medio Oriente,0.995
4566,Jair Bolsonaro,Lula Da Silva,0.994
